# 作业3 - 深度学习卷积神经网络

**课程**: 深度学习
**学号**: 20234080327
**姓名**: 赵果剑

---
## 2. 卷积和池化层
### 2.1 理论计算题

**已知**:
- 输入图像尺寸: $3 \times 32 \times 32$ (通道数 $\times$ 高 $\times$ 宽)
- 卷积层: 16 个卷积核, 每个卷积核 $3 \times 5 \times 5$
- Padding = 2, Stride = 2

**1. 输出特征图尺寸**:

输出特征图尺寸公式:
$$O = \left\lfloor \frac{H + 2P - K}{S} \right\rfloor + 1$$

其中 $H=32$, $K=5$, $P=2$, $S=2$:

$$O_{height} = \left\lfloor \frac{32 + 2 \times 2 - 5}{2} \right\rfloor + 1
= \left\lfloor \frac{31}{2} \right\rfloor + 1
= 15 + 1 = 16$$

$$O_{width} = \left\lfloor \frac{32 + 2 \times 2 - 5}{2} \right\rfloor + 1 = 16$$

**输出特征图尺寸: $16 \times 16 \times 16$ (通道数 $\times$ 高 $\times$ 宽)**

其中:
- 输出通道数 = 卷积核数量 = 16
- 输出高度 = 16
- 输出宽度 = 16

**2. 单输出像素所需的乘法次数**:

每个输出像素对应一次完整的卷积操作:
- 卷积核大小: $3 \times 5 \times 5 = 75$
- 每次卷积操作中, 每个权重与对应输入值相乘
- 乘法操作次数 = 卷积核参数数量 = $3 \times 5 \times 5 = 75$

**答: 每个输出像素需要 75 次乘法操作**

In [3]:
# 2.1 验证计算
import numpy as np

# 参数
C_in = 3
H_in = 32
W_in = 32
C_out = 16
K = 5
padding = 2
stride = 2

# 计算输出尺寸
H_out = (H_in + 2 * padding - K) // stride + 1
W_out = (W_in + 2 * padding - K) // stride + 1

print(f"输出特征图尺寸: {C_out} × {H_out} × {W_out}")
print(f"每个输出像素的乘法次数: {C_in} × {K} × {K} = {C_in * K * K}")

输出特征图尺寸: 16 × 16 × 16
每个输出像素的乘法次数: 3 × 5 × 5 = 75


### 2.2 编程题 — 手动实现 Max Pooling 前向传播

使用 Python 和 NumPy 手动实现支持 stride 和 padding 的二维最大池化前向传播函数。

In [4]:
import numpy as np

def max_pool2d_forward(x, kernel_size, stride=None, padding=0):
    """
    二维最大池化前向传播 (手动实现)
    
    参数:
        x: 输入张量, shape=(N, C, H, W) 或 (C, H, W)
        kernel_size: 池化窗口大小, int 或 (kh, kw)
        stride: 步幅, int 或 (sh, sw), 默认等于 kernel_size
        padding: 填充, int 或 (ph, pw), 默认 0
    
    返回:
        out: 池化后的输出张量
    """
    # 参数标准化
    if isinstance(kernel_size, int):
        k_h, k_w = kernel_size, kernel_size
    else:
        k_h, k_w = kernel_size
    
    if stride is None:
        stride_h, stride_w = k_h, k_w
    elif isinstance(stride, int):
        stride_h, stride_w = stride, stride
    else:
        stride_h, stride_w = stride
    
    if isinstance(padding, int):
        pad_h, pad_w = padding, padding
    else:
        pad_h, pad_w = padding
    
    # 确定输入维度并处理 batch 维度
    if x.ndim == 3:
        # (C, H, W) -> (1, C, H, W)
        x = x[np.newaxis, :]
        is_batched = False
    elif x.ndim == 4:
        is_batched = True
    else:
        raise ValueError(f"输入维度应为 3 或 4, 但得到了 {x.ndim}")
    
    N, C, H, W = x.shape
    
    # 对输入进行填充
    if pad_h > 0 or pad_w > 0:
        x_padded = np.pad(x, 
                          ((0, 0), (0, 0), (pad_h, pad_h), (pad_w, pad_w)),
                          mode='constant', constant_values=float('-inf'))
    else:
        x_padded = x
    
    H_pad, W_pad = x_padded.shape[2], x_padded.shape[3]
    
    # 计算输出尺寸
    H_out = (H_pad - k_h) // stride_h + 1
    W_out = (W_pad - k_w) // stride_w + 1
    
    # 初始化输出
    out = np.zeros((N, C, H_out, W_out))
    
    # 滑动窗口计算最大池化
    for i in range(H_out):
        for j in range(W_out):
            h_start = i * stride_h
            h_end = h_start + k_h
            w_start = j * stride_w
            w_end = w_start + k_w
            
            # 在池化窗口内取最大值
            window = x_padded[:, :, h_start:h_end, w_start:w_end]
            out[:, :, i, j] = np.max(window, axis=(2, 3))
    
    if not is_batched:
        out = out.squeeze(0)
    
    return out


# 测试代码
if __name__ == "__main__":
    # 测试 1: 基本 2x2 池化, stride=2, padding=0
    print("=" * 60)
    print("测试 1: 4x4 输入, 2x2 池化, stride=2, padding=0")
    x1 = np.array([[1, 2, 3, 4],
                    [5, 6, 7, 8],
                    [9, 10, 11, 12],
                    [13, 14, 15, 16]], dtype=float).reshape(1, 1, 4, 4)
    out1 = max_pool2d_forward(x1, kernel_size=2, stride=2, padding=0)
    print(f"输入:\n{x1[0, 0]}")
    print(f"输出:\n{out1[0, 0]}")
    print(f"期望输出 (对角):\n[[6, 8],\\n [14, 16]]")
    
    # 测试 2: 带 padding
    print("\n" + "=" * 60)
    print("测试 2: 3x3 输入, 2x2 池化, stride=1, padding=1")
    x2 = np.array([[1, 2, 3],
                    [4, 5, 6],
                    [7, 8, 9]], dtype=float).reshape(1, 1, 3, 3)
    out2 = max_pool2d_forward(x2, kernel_size=2, stride=1, padding=1)
    print(f"输入:\n{x2[0, 0]}")
    print(f"输出 shape: {out2.shape}")
    print(f"输出:\n{out2[0, 0]}")
    
    # 验证与 PyTorch 的对比
    try:
        import torch
        import torch.nn.functional as F
        
        print("\n" + "=" * 60)
        print("测试 3: 与 PyTorch MaxPool2d 对比")
        
        torch.manual_seed(42)
        x_torch = torch.randn(2, 3, 8, 8)
        x_numpy = x_torch.numpy()
        
        # PyTorch MaxPool2d
        torch_out = F.max_pool2d(x_torch, kernel_size=3, stride=2, padding=1)
        
        # 手动实现
        custom_out = max_pool2d_forward(x_numpy, kernel_size=3, stride=2, padding=1)
        
        diff = np.abs(torch_out.numpy() - custom_out).max()
        print(f"PyTorch 输出 shape: {torch_out.shape}")
        print(f"自定义输出 shape: {custom_out.shape}")
        print(f"最大差异: {diff:.10f}")
        print(f"结果一致: {diff < 1e-5}")
    except ImportError:
        print("PyTorch 未安装, 跳过对比测试")
    
    print("\n" + "=" * 60)
    print("所有测试完成!")

测试 1: 4x4 输入, 2x2 池化, stride=2, padding=0
输入:
[[ 1.  2.  3.  4.]
 [ 5.  6.  7.  8.]
 [ 9. 10. 11. 12.]
 [13. 14. 15. 16.]]
输出:
[[ 6.  8.]
 [14. 16.]]
期望输出 (对角):
[[6, 8],\n [14, 16]]

测试 2: 3x3 输入, 2x2 池化, stride=1, padding=1
输入:
[[1. 2. 3.]
 [4. 5. 6.]
 [7. 8. 9.]]
输出 shape: (1, 1, 4, 4)
输出:
[[1. 2. 3. 3.]
 [4. 5. 6. 6.]
 [7. 8. 9. 9.]
 [7. 8. 9. 9.]]

测试 3: 与 PyTorch MaxPool2d 对比
PyTorch 输出 shape: torch.Size([2, 3, 4, 4])
自定义输出 shape: (2, 3, 4, 4)
最大差异: 0.0000000000
结果一致: True

所有测试完成!


---
## 3. LeNet, AlexNet, VGG 和 NiN
### 3.1 理论计算题

**已知**:
- 输入和输出特征图的通道数均为 $C$
- 卷积层不带偏置 (bias)

**1. 一个 $5 \times 5$ 卷积层的参数量**:

$$\text{参数量} = C \times C \times 5 \times 5 = 25C^2$$

其中:
- 每个卷积核: $C \times 5 \times 5$ (输入通道 $\times$ 高 $\times$ 宽)
- 共 $C$ 个卷积核 (输出通道数 = C)
- 无偏置

**2. 两个串联的 $3 \times 3$ 卷积层的总参数量**:

第一层: $C \times C \times 3 \times 3 = 9C^2$

第二层: $C \times C \times 3 \times 3 = 9C^2$

总参数量: $9C^2 + 9C^2 = 18C^2$

**结论**: 两个 $3 \times 3$ 卷积层 ($18C^2$) 的参数量少于一个 $5 \times 5$ 卷积层 ($25C^2$), 同时拥有更大的感受野 ($5 \times 5$), 体现了 VGG 设计的优势。

In [5]:
# 3.1 验证计算
C = 64  # 示例通道数

params_5x5 = C * C * 5 * 5
params_3x3_1 = C * C * 3 * 3
params_3x3_2 = C * C * 3 * 3
total_3x3 = params_3x3_1 + params_3x3_2

print(f"C = {C} 时:")
print(f"一个 5×5 卷积层参数量: {params_5x5:,}")
print(f"两个 3×3 卷积层总参数量: {total_3x3:,}")
print(f"参数量减少比例: {(1 - total_3x3/params_5x5) * 100:.1f}%")

C = 64 时:
一个 5×5 卷积层参数量: 102,400
两个 3×3 卷积层总参数量: 73,728
参数量减少比例: 28.0%


### 3.2 编程题 — NiN Block

使用 PyTorch 的 `torch.nn.Sequential` 定义一个标准的 NiN 块。

In [6]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    """
    NiN (Network in Network) 块
    
    包含: 一个普通卷积层 + 两个 1×1 卷积层
    每层卷积后紧跟 ReLU 激活
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super(NiNBlock, self).__init__()
        self.net = nn.Sequential(
            # 普通卷积层 (指定 kernel_size, stride, padding)
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(inplace=True),
            # 第一个 1×1 卷积 (MLP卷积)  — 等价于全连接层的替换
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(inplace=True),
            # 第二个 1×1 卷积
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x):
        return self.net(x)


# 测试 NiN Block
if __name__ == "__main__":
    # 创建 NiN 块
    nin_block = NiNBlock(in_channels=3, out_channels=96, 
                         kernel_size=11, stride=4, padding=2)
    
    print("NiN Block 结构:")
    print(nin_block)
    
    # 前向传播测试
    x = torch.randn(1, 3, 224, 224)
    out = nin_block(x)
    print(f"\n输入 shape: {x.shape}")
    print(f"输出 shape: {out.shape}")

NiN Block 结构:
NiNBlock(
  (net): Sequential(
    (0): Conv2d(3, 96, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
    (3): ReLU(inplace=True)
    (4): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
    (5): ReLU(inplace=True)
  )
)

输入 shape: torch.Size([1, 3, 224, 224])
输出 shape: torch.Size([1, 96, 55, 55])


---
## 4. Inception, 批量归一化和残差网络
### 4.1 理论计算题 — Batch Normalization

**已知**:
- 小批量特征值: $x_1 = 2, x_2 = 4, x_3 = 6, x_4 = 8$
- 缩放参数 $\gamma = 2$, 平移参数 $\beta = 1$, 常数 $\epsilon = 0$

**计算步骤**:

**1. 计算批量均值 $\mu$**:
$$\mu = \frac{1}{m} \sum_{i=1}^{m} x_i = \frac{2 + 4 + 6 + 8}{4} = 5$$

**2. 计算批量方差 $\sigma^2$** (使用总体方差, 除 N):
$$\sigma^2 = \frac{1}{m} \sum_{i=1}^{m} (x_i - \mu)^2$$
$$= \frac{(2-5)^2 + (4-5)^2 + (6-5)^2 + (8-5)^2}{4}$$
$$= \frac{9 + 1 + 1 + 9}{4} = 5$$

**3. 计算归一化输出**:
$$y_i = \gamma \cdot \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

代入 $\gamma = 2, \beta = 1, \epsilon = 0$:

$$y_1 = 2 \times \frac{2 - 5}{\sqrt{5}} + 1 = -\frac{6}{\sqrt{5}} + 1 \approx -2.6832 + 1 = -1.6832$$

$$y_2 = 2 \times \frac{4 - 5}{\sqrt{5}} + 1 = -\frac{2}{\sqrt{5}} + 1 \approx -0.8944 + 1 = 0.1056$$

$$y_3 = 2 \times \frac{6 - 5}{\sqrt{5}} + 1 = \frac{2}{\sqrt{5}} + 1 \approx 0.8944 + 1 = 1.8944$$

$$y_4 = 2 \times \frac{8 - 5}{\sqrt{5}} + 1 = \frac{6}{\sqrt{5}} + 1 \approx 2.6832 + 1 = 3.6832$$

**精确值**:
$$y_1 = 1 - \frac{6}{\sqrt{5}}, \quad y_2 = 1 - \frac{2}{\sqrt{5}}, \quad y_3 = 1 + \frac{2}{\sqrt{5}}, \quad y_4 = 1 + \frac{6}{\sqrt{5}}$$

In [7]:
# 4.1 验证 Batch Normalization 的计算
import numpy as np

x = np.array([2.0, 4.0, 6.0, 8.0])
gamma = 2.0
beta = 1.0
eps = 0.0

mu = np.mean(x)
var = np.var(x)  # 总体方差 (除 N)

y = gamma * (x - mu) / np.sqrt(var + eps) + beta

print(f"输入 x: {x}")
print(f"均值 μ = {mu:.4f}")
print(f"方差 σ² = {var:.4f}")
print(f"标准差 σ = {np.sqrt(var):.4f}")
print()
for i, (xi, yi) in enumerate(zip(x, y)):
    print(f"y{i+1} = {yi:.6f}")

输入 x: [2. 4. 6. 8.]
均值 μ = 5.0000
方差 σ² = 5.0000
标准差 σ = 2.2361

y1 = -1.683282
y2 = 0.105573
y3 = 1.894427
y4 = 3.683282


### 4.2 编程题 — 残差块 (Residual Block)

用 PyTorch 自定义残差块类 `Residual`, 包含两个 3×3 卷积层, 每层后跟批量归一化。

In [8]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    """
    残差块 (Residual Block)
    
    包含两个具有相同输出通道数的 3×3 卷积层,
    每个卷积层后跟一个批量归一化层。
    如果 use_1x1conv=True, 则使用 1×1 卷积调整输入以匹配输出形状。
    """
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super(Residual, self).__init__()
        
        # 第一个 3×3 卷积 + BN + ReLU
        self.conv1 = nn.Conv2d(in_channels, out_channels, 
                               kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        # 第二个 3×3 卷积 + BN
        self.conv2 = nn.Conv2d(out_channels, out_channels,
                               kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 1×1 卷积 (可选) — 用于调整输入通道数和形状
        self.use_1x1conv = use_1x1conv
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, out_channels,
                                   kernel_size=1, stride=stride, bias=False)
            self.bn3 = nn.BatchNorm2d(out_channels)
        
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        identity = x
        
        # 第一个卷积 + BN + ReLU
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        # 第二个卷积 + BN
        out = self.conv2(out)
        out = self.bn2(out)
        
        # 如果需要, 用 1×1 卷积调整输入
        if self.use_1x1conv:
            identity = self.conv3(x)
            identity = self.bn3(identity)
        
        # 残差连接: f(x) + x
        out += identity
        out = self.relu(out)
        
        return out


# 测试 Residual Block
if __name__ == "__main__":
    print("=" * 60)
    print("测试 1: 残差块 (use_1x1conv=False, 输入输出通道相同)")
    res_block1 = Residual(in_channels=64, out_channels=64, use_1x1conv=False)
    x1 = torch.randn(1, 64, 32, 32)
    y1 = res_block1(x1)
    print(f"输入 shape: {x1.shape}")
    print(f"输出 shape: {y1.shape}")
    
    print("\n" + "=" * 60)
    print("测试 2: 残差块 (use_1x1conv=True, 通道数改变)")
    res_block2 = Residual(in_channels=64, out_channels=128, 
                          use_1x1conv=True, stride=2)
    x2 = torch.randn(1, 64, 32, 32)
    y2 = res_block2(x2)
    print(f"输入 shape: {x2.shape}")
    print(f"输出 shape: {y2.shape}")
    
    print("\n残差块结构:")
    print(res_block2)

测试 1: 残差块 (use_1x1conv=False, 输入输出通道相同)
输入 shape: torch.Size([1, 64, 32, 32])
输出 shape: torch.Size([1, 64, 32, 32])

测试 2: 残差块 (use_1x1conv=True, 通道数改变)
输入 shape: torch.Size([1, 64, 32, 32])
输出 shape: torch.Size([1, 128, 16, 16])

残差块结构:
Residual(
  (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(64, 128, kernel_size=(1, 1), stride=(2, 2), bias=False)
  (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
)


---
## 5. 图像增广, 微调和样式迁移
### 5.1 理论计算题 — Fine-tuning

**1. 为什么底层特征提取层用小学习率 (或冻结), 顶层输出层用大学习率?**

> 底层卷积层学习的是通用/低级特征 (如边缘、纹理、颜色斑点、角点等), 这些特征在大多数视觉任务中都是通用的, 预训练权重已经很好地学习了这些特征。因此, 使用较小的学习率可以避免破坏这些已经学好的通用特征 (防止灾难性遗忘), 仅做微小的调整即可。
>
> 顶层全连接层 (分类器) 是任务相关的, 需要适应新的目标类别。新初始化的输出层权重是随机的, 还没有学到任何有效特征, 因此需要较大的学习率来快速收敛, 以适应新任务的分类要求。

**2. 目标数据集非常小且与源数据集非常相似时的微调策略:**

> ① **冻结大部分底层特征提取层**: 保留预训练模型除输出层外的所有权重不变, 仅训练新初始化的分类层 (也称为线性探测, Linear Probing)。
>
> ② **仅替换和训练最后的全连接层**: 由于数据集小且与源数据相似, 底层特征已经足够好, 只需要学习新的类别映射。
>
> ③ **使用较小的学习率**: 如果确实需要微调少量层, 学习率应远小于从头训练时的学习率 (如 1/10 或更小)。
>
> ④ **增加正则化**: 使用更强的权重衰减 (weight decay)、Dropout 等正则化手段防止过拟合。
>
> ⑤ **数据增强**: 使用图像增广 (随机裁剪、翻转、颜色抖动等) 来增加训练数据的多样性。

### 5.2 编程题 — 图像增广 Pipeline

使用 `torchvision.transforms` 创建组合图像增广管道。

In [9]:
import torchvision.transforms as transforms
from PIL import Image
import numpy as np

# 创建图像增广 Pipeline
augmentation_pipeline = transforms.Compose([
    # 1. 随机裁剪, 面积比例 0.08 到 1.0, 缩放到 224x224
    transforms.RandomResizedCrop(
        size=224,
        scale=(0.08, 1.0),
        ratio=(0.75, 1.3333333333333333),  # 默认宽高比范围
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    # 2. 50% 概率水平翻转
    transforms.RandomHorizontalFlip(p=0.5),
    # 3. 随机改变亮度、对比度、饱和度, 变化范围 0.5
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0),
    # 4. 转换为 PyTorch Tensor
    transforms.ToTensor(),
])

print("图像增广 Pipeline 创建成功!")
print(f"Pipeline 包含 {len(augmentation_pipeline.transforms)} 个步骤:")
for i, t in enumerate(augmentation_pipeline.transforms, 1):
    print(f"  {i}. {type(t).__name__}: {t}")

图像增广 Pipeline 创建成功!
Pipeline 包含 4 个步骤:
  1. RandomResizedCrop: RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
  2. RandomHorizontalFlip: RandomHorizontalFlip(p=0.5)
  3. ColorJitter: ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
  4. ToTensor: ToTensor()


In [10]:
# 测试 Pipeline (用随机图像)

# 创建一张随机测试图像 (RGB, 256x256)
test_img = Image.fromarray(
    np.random.randint(0, 256, (256, 256, 3), dtype=np.uint8)
)
print(f"原始图像大小: {test_img.size}")

# 应用 Pipeline
augmented_tensor = augmentation_pipeline(test_img)
print(f"增强后 Tensor shape: {augmented_tensor.shape}")
print(f"Tensor 数据类型: {augmented_tensor.dtype}")
print(f"像素值范围: [{augmented_tensor.min():.4f}, {augmented_tensor.max():.4f}]")

# 显示多次增强的效果
print("\n应用多次增强 (显示图像尺寸和统计信息):")
for i in range(3):
    result = augmentation_pipeline(test_img)
    print(f"  第 {i+1} 次: shape={result.shape}, "
          f"mean={result.mean():.4f}, std={result.std():.4f}")

原始图像大小: (256, 256)
增强后 Tensor shape: torch.Size([3, 224, 224])
Tensor 数据类型: torch.float32
像素值范围: [0.1216, 0.9255]

应用多次增强 (显示图像尺寸和统计信息):
  第 1 次: shape=torch.Size([3, 224, 224]), mean=0.6188, std=0.2340
  第 2 次: shape=torch.Size([3, 224, 224]), mean=0.4035, std=0.1765
  第 3 次: shape=torch.Size([3, 224, 224]), mean=0.2831, std=0.0996


---
## 6. 目标检测, 计算机视觉训练技巧
### 6.1 理论计算题 — IoU

**已知**:
- 真实框 A = [10, 10, 50, 50]
- 预测框 B = [30, 30, 70, 70]
- 格式: [左上角 x, 左上角 y, 右下角 x, 右下角 y]

**计算 IoU**:

$$\text{IoU} = \frac{\text{Intersection}}{\text{Union}}$$

**1. 计算交集区域**:

- 交集左上角 x: $\max(10, 30) = 30$
- 交集左上角 y: $\max(10, 30) = 30$
- 交集右下角 x: $\min(50, 70) = 50$
- 交集右下角 y: $\min(50, 50) = 50$  ← A 的右下角 y = 50
- 交集宽: $50 - 30 = 20$
- 交集高: $50 - 30 = 20$
- 交集面积: $20 \times 20 = 400$

**2. 计算各框面积**:

- A 的面积: $(50 - 10) \times (50 - 10) = 40 \times 40 = 1600$
- B 的面积: $(70 - 30) \times (70 - 30) = 40 \times 40 = 1600$

**3. 计算并集面积**:

$$\text{Union} = \text{Area}_A + \text{Area}_B - \text{Intersection}$$
$$= 1600 + 1600 - 400 = 2800$$

**4. 计算 IoU**:

$$\text{IoU} = \frac{400}{2800} = \frac{1}{7} \approx 0.1429$$

**答: IoU = 1/7 ≈ 0.1429**

In [11]:
# 6.1 验证 IoU 计算

def compute_iou(box_a, box_b):
    """计算两个边界框的 IoU
    格式: [x1, y1, x2, y2] (左上角 x, 左上角 y, 右下角 x, 右下角 y)
    """
    # 计算交集坐标
    inter_x1 = max(box_a[0], box_b[0])
    inter_y1 = max(box_a[1], box_b[1])
    inter_x2 = min(box_a[2], box_b[2])
    inter_y2 = min(box_a[3], box_b[3])
    
    # 交集面积
    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h
    
    # 各框面积
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    
    # 并集面积
    union_area = area_a + area_b - inter_area
    
    # IoU
    iou = inter_area / union_area if union_area > 0 else 0.0
    
    return iou, inter_area, union_area


# 给定框
A = [10, 10, 50, 50]
B = [30, 30, 70, 70]

iou, inter_area, union_area = compute_iou(A, B)

print(f"真实框 A: {A}")
print(f"预测框 B: {B}")
print(f"A 面积: {40*40}")
print(f"B 面积: {40*40}")
print(f"交集面积: {inter_area}")
print(f"并集面积: {union_area}")
print(f"IoU = {iou} = {iou:.4f}")
print(f"IoU (分数形式): {int(inter_area)}/{int(union_area)} = {inter_area/union_area:.4f}")

真实框 A: [10, 10, 50, 50]
预测框 B: [30, 30, 70, 70]
A 面积: 1600
B 面积: 1600
交集面积: 400
并集面积: 2800
IoU = 0.14285714285714285 = 0.1429
IoU (分数形式): 400/2800 = 0.1429


### 6.2 编程题 — 标签平滑交叉熵损失 (Label Smoothing)

实现标签平滑后的交叉熵损失函数。

平滑后的标签分布:
- 真实类别: $1 - \epsilon$
- 其他类别: $\frac{\epsilon}{K-1}$

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class LabelSmoothingCrossEntropy(nn.Module):
    """
    标签平滑交叉熵损失 (Label Smoothing Cross-Entropy Loss)
    
    公式:
        真实标签概率: 1 - epsilon
        其他标签概率: epsilon / (K - 1)
        其中 K 为类别数
    """
    def __init__(self, epsilon=0.1, reduction='mean'):
        super(LabelSmoothingCrossEntropy, self).__init__()
        self.epsilon = epsilon
        self.reduction = reduction
    
    def forward(self, logits, targets):
        """
        参数:
            logits: 模型输出 (未经过 softmax), shape=(N, K)
            targets: 真实标签索引, shape=(N,), 取值范围 [0, K-1]
        返回:
            损失值
        """
        K = logits.size(-1)  # 类别数
        
        # 创建平滑后的标签分布
        # shape: (N, K)
        smooth_labels = torch.full_like(logits, self.epsilon / (K - 1))
        
        # 将真实类别位置的概率设为 1 - epsilon
        smooth_labels.scatter_(1, targets.unsqueeze(1), 1 - self.epsilon)
        
        # 计算 log softmax
        log_probs = F.log_softmax(logits, dim=1)
        
        # 计算交叉熵损失
        loss = -(smooth_labels * log_probs).sum(dim=1)
        
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss


# 测试标签平滑损失
if __name__ == "__main__":
    print("=" * 60)
    print("标签平滑交叉熵损失测试")
    
    # 创建损失函数
    epsilon = 0.1
    criterion_smooth = LabelSmoothingCrossEntropy(epsilon=epsilon)
    criterion_ce = nn.CrossEntropyLoss()
    
    # 测试数据
    torch.manual_seed(42)
    N, K = 4, 10  # 4个样本, 10分类
    logits = torch.randn(N, K)
    targets = torch.randint(0, K, (N,))
    
    # 计算两种损失
    loss_smooth = criterion_smooth(logits, targets)
    loss_ce = criterion_ce(logits, targets)
    
    print(f"batch_size={N}, num_classes={K}, epsilon={epsilon}")
    print(f"真实标签: {targets.tolist()}")
    print()
    print(f"标准交叉熵损失: {loss_ce.item():.6f}")
    print(f"标签平滑交叉熵损失: {loss_smooth.item():.6f}")
    print(f"差异: {loss_smooth.item() - loss_ce.item():.6f}")
    print("(标签平滑损失通常略大于标准交叉熵损失)")
    
    # 验证平滑标签分布
    print("\n" + "=" * 60)
    print("验证平滑标签分布 (第一个样本):")
    t = targets[0]  # 第一个样本的标签
    smooth_labels = torch.full((1, K), epsilon / (K - 1))
    smooth_labels[0, t] = 1 - epsilon
    print(f"目标类别: {t}")
    print(f"真实标签概率: {smooth_labels[0, t]:.4f}")
    print(f"其他类别概率: {smooth_labels[0, 0]:.4f}")
    print(f"标签和: {smooth_labels.sum():.4f} (应为 1.0)")

标签平滑交叉熵损失测试
batch_size=4, num_classes=10, epsilon=0.1
真实标签: [5, 4, 8, 8]

标准交叉熵损失: 3.472086
标签平滑交叉熵损失: 3.404533
差异: -0.067554
(标签平滑损失通常略大于标准交叉熵损失)

验证平滑标签分布 (第一个样本):
目标类别: 5
真实标签概率: 0.9000
其他类别概率: 0.0111
标签和: 1.0000 (应为 1.0)


In [13]:
# 6.2 (续) 函数版本 — 更简洁的实现

def label_smoothing_loss(logits, targets, epsilon=0.1):
    """
    标签平滑交叉熵损失函数
    
    参数:
        logits: 模型输出, shape=(N, K)
        targets: 真实标签索引, shape=(N,)
        epsilon: 平滑因子
    返回:
        标量损失值
    """
    K = logits.size(-1)
    
    # 使用 -log_softmax 计算
    log_probs = F.log_softmax(logits, dim=-1)
    
    # 标准交叉熵损失 (独热编码)
    nll_loss = F.nll_loss(log_probs, targets, reduction='none')
    
    # 标签平滑正则项
    smooth_loss = -log_probs.mean(dim=-1)
    
    # 组合损失
    loss = (1 - epsilon) * nll_loss + epsilon * smooth_loss
    
    return loss.mean()


# 测试函数版本
torch.manual_seed(42)
logits = torch.randn(4, 10)
targets = torch.randint(0, 10, (4,))

loss_fn = label_smoothing_loss(logits, targets, epsilon=0.1)
print(f"标签平滑损失 (函数版本): {loss_fn.item():.6f}")

# 与类版本对比
criterion = LabelSmoothingCrossEntropy(epsilon=0.1)
loss_cls = criterion(logits, targets)
print(f"标签平滑损失 (类版本):   {loss_cls.item():.6f}")
print(f"结果一致: {abs(loss_fn.item() - loss_cls.item()) < 1e-6}")

标签平滑损失 (函数版本): 3.411288
标签平滑损失 (类版本):   3.404533
结果一致: False
